# 02 Multi-Head Attention Shape Lab

以 `X` 为主线追踪 `[B,T,C] → [B,H,T,D] → [B,T,C]`。

In [1]:
import math
import torch
from torch import nn

torch.manual_seed(42)
B, T, C, H = 2, 6, 32, 4
D = C // H
X = torch.randn(B, T, C)
q_proj = nn.Linear(C, C)
k_proj = nn.Linear(C, C)
v_proj = nn.Linear(C, C)

Q = q_proj(X)
K = k_proj(X)
V = v_proj(X)
print("X/Q/K/V:", X.shape, Q.shape)

X/Q/K/V: torch.Size([2, 6, 32]) torch.Size([2, 6, 32])


In [2]:
def split_heads(x):
    # [B,T,C] -> [B,T,H,D] -> [B,H,T,D]
    return x.view(B, T, H, D).transpose(1, 2)

Qh = split_heads(Q)
Kh = split_heads(K)
Vh = split_heads(V)
print("Qh:", Qh.shape)

Qh: torch.Size([2, 4, 6, 8])


In [3]:
scores = Qh @ Kh.transpose(-2, -1) / math.sqrt(D)
print("scores:", scores.shape)  # [B,H,T,T]

causal = torch.ones(T, T, dtype=torch.bool).tril()
scores = scores.masked_fill(~causal, float("-inf"))
weights = torch.softmax(scores, dim=-1)
Yh = weights @ Vh
print("Y per head:", Yh.shape)

scores: torch.Size([2, 4, 6, 6])
Y per head: torch.Size([2, 4, 6, 8])


In [4]:
# [B,H,T,D] -> [B,T,H,D] -> [B,T,C]
Y = Yh.transpose(1, 2).contiguous().view(B, T, C)
print("final Y:", Y.shape)

final Y: torch.Size([2, 6, 32])


## 为什么 Head 不是“把不同 token 分组”？

Head 切分的是 **每个 token 的特征维度 `C`**：

```text
C = H × D
```

每个 Head 仍然同时看到整条序列，只是在不同子空间里学习不同的 Q/K/V 关系。